# 🤖 Forex Trading Bot — ML Based
### Running di Google Colab

Notebook ini menjalankan seluruh stack:
- **Redis** — message broker untuk Celery
- **Celery Worker** — background task processor
- **FastAPI** — REST API (port 8000)
- **Flask Web Dashboard** — UI dashboard (port 5000)
- **ngrok** — expose port supaya bisa diakses dari browser

> Jalankan setiap cell secara berurutan dari atas ke bawah.

---
## 1. Clone Repository

In [1]:
import os

REPO_DIR = "/content/forex_trading_bot_ml_based"

if os.path.exists(REPO_DIR):
    print("Repo sudah ada, pull latest changes...")
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Herutriana44/forex_trading_bot_ml_based.git {REPO_DIR}
    print("Clone selesai!")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/forex_trading_bot_ml_based'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 67 (delta 11), reused 62 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 71.90 KiB | 3.13 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Clone selesai!
Working directory: /content/forex_trading_bot_ml_based


In [7]:
!git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.08 KiB | 550.00 KiB/s, done.
From https://github.com/Herutriana44/forex_trading_bot_ml_based
   e243f76..fb02b8c  main       -> origin/main
Updating e243f76..fb02b8c
Fast-forward
 src/db/logging.py | 8 ++++----
 src/web/app.py    | 2 +-
 2 files changed, 5 insertions(+), 5 deletions(-)


---
## 2. Install Dependencies

In [2]:
# Install semua Python dependencies dari requirements.txt
!pip install -q -r requirements.txt

# Install Flask + SocketIO untuk web dashboard
!pip install -q flask flask-socketio

# Install pyngrok untuk expose port ke internet
!pip install -q pyngrok

print("✅ Semua dependencies berhasil diinstall.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
✅ Semua dependencies berhasil diinstall.


---
## 3. Install & Jalankan Redis

In [3]:
# Install Redis di Colab (Ubuntu-based)
!apt-get install -qq redis-server > /dev/null 2>&1

# Jalankan Redis di background
!redis-server --daemonize yes

import time
time.sleep(2)

# Verifikasi Redis berjalan
result = !redis-cli ping
if 'PONG' in result[0]:
    print("✅ Redis berjalan dengan baik!")
else:
    print("❌ Redis gagal start:", result)

✅ Redis berjalan dengan baik!


---
## 4. Setup Environment Variables

In [4]:
import os

# Pastikan working directory benar
os.chdir("/content/forex_trading_bot_ml_based")

# Set environment variables
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["CELERY_RESULT_BACKEND"] = "redis://localhost:6379/1"
os.environ["DATABASE_URL"] = "sqlite:////content/forex_trading_bot_ml_based/src/db/trading.db"
os.environ["RETRAIN_ACCURACY_THRESHOLD"] = "0.55"
os.environ["PYTHONPATH"] = "/content/forex_trading_bot_ml_based"

# Buat folder models jika belum ada
os.makedirs("src/models/versioned", exist_ok=True)
os.makedirs("src/models/current", exist_ok=True)
os.makedirs("src/db", exist_ok=True)

print("✅ Environment variables berhasil di-set.")

✅ Environment variables berhasil di-set.


---
## 5. Inisialisasi Database

In [8]:
import sys
sys.path.insert(0, "/content/forex_trading_bot_ml_based")

# Import dan inisialisasi database (buat tabel jika belum ada)
from src.db.logging import Base, engine
Base.metadata.create_all(bind=engine)

print("✅ Database berhasil diinisialisasi.")

✅ Database berhasil diinisialisasi.


---
## 6. Training Model Pertama Kali

> Wajib dijalankan sebelum prediksi. Proses download data dari Yahoo Finance dan melatih model XGBoost.

In [11]:
from src.retraining.pipeline import retrain_models

print("\u23f3 Memulai training model untuk EURUSD=X...")
print("   (Download data Yahoo Finance + feature engineering + training 3 model)")
print()

results = retrain_models(symbol="EURUSD=X", start_date="2019-01-01")

best = results["best_model"]
print()
print("\u2705 Training selesai!")
print(f"   Best Model : {best['name']}")
print(f"   Akurasi    : {best['metrics']['accuracy']:.4f}")
print(f"   Precision  : {best['metrics']['precision']:.4f}")
print(f"   Recall     : {best['metrics']['recall']:.4f}")
print(f"   Dipromote  : {results['promoted']}")
print()
print("\U0001f4ca Semua hasil model:")
for r in results['all_results']:
    print(f"   {r['model_name']:30s}  acc={r['metrics']['accuracy']:.4f}  prec={r['metrics']['precision']:.4f}  rec={r['metrics']['recall']:.4f}")

⏳ Memulai training model untuk EURUSD=X...
   (Download data Yahoo Finance + feature engineering + training 3 model)

Mengunduh data EURUSD=X dari 2019-01-01 sampai 2026-06-19...


/content/forex_trading_bot_ml_based/src/inference/feature_engineer.py:87: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Membuat fitur teknikal...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...

Best model: Random Forest (acc=0.5277)
Accuracy 0.5277 below threshold 0.55. Not promoted.

✅ Training selesai!
   Best Model : Random Forest
   Akurasi    : 0.5277
   Precision  : 0.4896
   Recall     : 0.2655
   Dipromote  : False

📊 Semua hasil model:
   Random Forest                   acc=0.5277  prec=0.4896  rec=0.2655
   Gradient Boosting               acc=0.5145  prec=0.4679  rec=0.2881
   XGBoost                         acc=0.5224  prec=0.4737  rec=0.2034


---
## 7. Jalankan Celery Worker (Background)

In [12]:
import subprocess, time

celery_proc = subprocess.Popen(
    [
        "python", "-m", "celery",
        "-A", "src.tasks.celery_app", "worker",
        "--loglevel=info",
        "--concurrency=2"
    ],
    cwd="/content/forex_trading_bot_ml_based",
    env={**os.environ, "PYTHONPATH": "/content/forex_trading_bot_ml_based"},
    stdout=open("/tmp/celery.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

if celery_proc.poll() is None:
    print(f"✅ Celery Worker berjalan. PID: {celery_proc.pid}")
else:
    print("❌ Celery Worker gagal start. Cek log di bawah:")
    !cat /tmp/celery.log

✅ Celery Worker berjalan. PID: 5087


---
## 8. Jalankan FastAPI Server (Background)

In [13]:
import subprocess, time

fastapi_proc = subprocess.Popen(
    [
        "python", "-m", "uvicorn",
        "src.api.main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    cwd="/content/forex_trading_bot_ml_based",
    env={**os.environ, "PYTHONPATH": "/content/forex_trading_bot_ml_based"},
    stdout=open("/tmp/fastapi.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(4)

if fastapi_proc.poll() is None:
    print(f"✅ FastAPI Server berjalan di port 8000. PID: {fastapi_proc.pid}")
else:
    print("❌ FastAPI gagal start. Cek log di bawah:")
    !cat /tmp/fastapi.log

✅ FastAPI Server berjalan di port 8000. PID: 5118


---
## 9. Jalankan Flask Web Dashboard (Background)

In [14]:
import subprocess, time

flask_proc = subprocess.Popen(
    ["python", "-m", "src.web.app"],
    cwd="/content/forex_trading_bot_ml_based",
    env={**os.environ, "PYTHONPATH": "/content/forex_trading_bot_ml_based"},
    stdout=open("/tmp/flask.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(4)

if flask_proc.poll() is None:
    print(f"✅ Flask Dashboard berjalan di port 5000. PID: {flask_proc.pid}")
else:
    print("❌ Flask gagal start. Cek log di bawah:")
    !cat /tmp/flask.log

❌ Flask gagal start. Cek log di bawah:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/forex_trading_bot_ml_based/src/web/app.py", line 203, in <module>
    socketio.run(app, debug=True, host='0.0.0.0', port=5000)
  File "/usr/local/lib/python3.12/dist-packages/flask_socketio/__init__.py", line 664, in run
    raise RuntimeError('The Werkzeug web server is not '
RuntimeError: The Werkzeug web server is not designed to run in production. Pass allow_unsafe_werkzeug=True to the run() method to disable this error.


---
## 10. Expose ke Internet via ngrok

> Daftarkan akun gratis di [ngrok.com](https://ngrok.com) dan dapatkan authtoken di https://dashboard.ngrok.com/get-started/your-authtoken

In [16]:
from pyngrok import ngrok
from google.colab import userdata

# ⚠️ Ganti dengan authtoken kamu dari https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Expose FastAPI (port 8000)
api_tunnel = ngrok.connect(8000)
print(f"🌐 FastAPI (REST API) : {api_tunnel.public_url}")
print(f"   Swagger Docs       : {api_tunnel.public_url}/docs")
print(f"   ReDoc              : {api_tunnel.public_url}/redoc")
print()

# Expose Flask Dashboard (port 5000)
web_tunnel = ngrok.connect(5000)
print(f"🖥️  Flask Dashboard    : {web_tunnel.public_url}")

🌐 FastAPI (REST API) : https://bb61-34-74-100-41.ngrok-free.app
   Swagger Docs       : https://bb61-34-74-100-41.ngrok-free.app/docs
   ReDoc              : https://bb61-34-74-100-41.ngrok-free.app/redoc

🖥️  Flask Dashboard    : https://2fdf-34-74-100-41.ngrok-free.app


---
## 11. Test API — Cek Status Model

In [17]:
import requests, json

BASE_URL = "http://localhost:8000/api/v1"

# Cek status model
response = requests.get(f"{BASE_URL}/model/status")
print("📊 Model Status:")
print(json.dumps(response.json(), indent=2))

📊 Model Status:
{
  "detail": "No current model found. Train a model first."
}


---
## 12. Test API — Buat Prediksi

In [18]:
import requests, json, time

BASE_URL = "http://localhost:8000/api/v1"

# Kirim request prediksi
symbol = "EURUSD=X"  # Ganti sesuai kebutuhan: GBPUSD=X, USDJPY=X, dll.

print(f"⏳ Membuat prediksi untuk {symbol}...")
resp = requests.post(f"{BASE_URL}/predict", json={"symbol": symbol})
task_info = resp.json()
task_id = task_info["task_id"]
print(f"   Task ID: {task_id}")

# Polling sampai hasil tersedia
for attempt in range(30):
    time.sleep(3)
    result = requests.get(f"{BASE_URL}/predict/{task_id}").json()
    status = result.get("status")
    print(f"   [{attempt+1}] Status: {status}")

    if status == "success":
        print()
        print("✅ Hasil Prediksi:")
        print(json.dumps(result, indent=2))
        break
    elif status == "error":
        print("❌ Error:", result.get("error"))
        break
else:
    print("⚠️ Timeout menunggu hasil prediksi.")

⏳ Membuat prediksi untuk EURUSD=X...
   Task ID: 6fcb93ee-a0af-49d5-b785-6dca637bd9c1
   [1] Status: pending
   [2] Status: pending
   [3] Status: pending
   [4] Status: pending
   [5] Status: pending
   [6] Status: pending
   [7] Status: pending
   [8] Status: pending
   [9] Status: pending
   [10] Status: pending
   [11] Status: pending
   [12] Status: pending
   [13] Status: pending
   [14] Status: pending
   [15] Status: pending
   [16] Status: pending
   [17] Status: pending
   [18] Status: pending
   [19] Status: pending
   [20] Status: pending
   [21] Status: pending
   [22] Status: pending
   [23] Status: pending
   [24] Status: pending
   [25] Status: pending
   [26] Status: pending
   [27] Status: pending
   [28] Status: pending
   [29] Status: pending
   [30] Status: pending
⚠️ Timeout menunggu hasil prediksi.


---
## 13. Test API — Trigger Retraining Model

In [19]:
import requests, json, time

BASE_URL = "http://localhost:8000/api/v1"

symbol = "EURUSD=X"
start_date = "2019-01-01"

print(f"⏳ Memulai retraining untuk {symbol} dari {start_date}...")
resp = requests.post(
    f"{BASE_URL}/model/retrain",
    json={"symbol": symbol, "start_date": start_date}
)
task_info = resp.json()
task_id = task_info["task_id"]
print(f"   Task ID: {task_id}")

# Polling sampai selesai
for attempt in range(40):
    time.sleep(5)
    result = requests.get(f"{BASE_URL}/model/retrain/{task_id}").json()
    status = result.get("status")
    print(f"   [{attempt+1}] Status: {status}")

    if status == "success":
        print()
        print("✅ Retraining selesai!")
        print(json.dumps(result, indent=2))
        break
    elif status == "error":
        print("❌ Error:", result.get("error"))
        break
else:
    print("⚠️ Timeout menunggu retraining selesai.")

⏳ Memulai retraining untuk EURUSD=X dari 2019-01-01...
   Task ID: 469519ca-7a8b-44ad-992d-66b02378cee1
   [1] Status: pending
   [2] Status: pending
   [3] Status: pending
   [4] Status: pending
   [5] Status: pending
   [6] Status: pending
   [7] Status: pending
   [8] Status: pending
   [9] Status: pending
   [10] Status: pending
   [11] Status: pending
   [12] Status: pending
   [13] Status: pending
   [14] Status: pending
   [15] Status: pending
   [16] Status: pending
   [17] Status: pending
   [18] Status: pending
   [19] Status: pending
   [20] Status: pending
   [21] Status: pending
   [22] Status: pending
   [23] Status: pending
   [24] Status: pending
   [25] Status: pending
   [26] Status: pending
   [27] Status: pending
   [28] Status: pending
   [29] Status: pending
   [30] Status: pending
   [31] Status: pending
   [32] Status: pending
   [33] Status: pending
   [34] Status: pending
   [35] Status: pending
   [36] Status: pending
   [37] Status: pending
   [38] Status: p

---
## 14. Cek Log Semua Service

In [20]:
print("===== LOG FASTAPI (50 baris terakhir) =====")
!tail -50 /tmp/fastapi.log

print("\n===== LOG CELERY (50 baris terakhir) =====")
!tail -50 /tmp/celery.log

print("\n===== LOG FLASK (50 baris terakhir) =====")
!tail -50 /tmp/flask.log

===== LOG FASTAPI (50 baris terakhir) =====
INFO:     127.0.0.1:44410 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:44426 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:38482 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:38490 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:38496 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:38508 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:54130 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:54144 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:54146 - "GET /api/v1/predict/6fcb93ee-a0af-49d5-b785-6dca637bd9c1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:54152 - "

---
## 15. Stop Semua Service

> Jalankan cell ini jika ingin mematikan semua proses.

In [21]:
from pyngrok import ngrok as _ngrok

# Tutup semua tunnel ngrok
try:
    _ngrok.kill()
    print("✅ Ngrok tunnels ditutup.")
except Exception as e:
    print(f"⚠️ Ngrok: {e}")

# Hentikan proses
for name, proc in [("Flask", flask_proc), ("FastAPI", fastapi_proc), ("Celery", celery_proc)]:
    try:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"✅ {name} dihentikan.")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

# Stop Redis
!redis-cli shutdown nosave 2>/dev/null || true
print("✅ Redis dihentikan.")

✅ Ngrok tunnels ditutup.
✅ Flask dihentikan.
✅ FastAPI dihentikan.
✅ Celery dihentikan.
✅ Redis dihentikan.
